#3 - A partir das extremidades das águas correntes e das águas correntes estimadas, traçar um círculo de 50m;

Desculpa a minha demora aqui... eu tive um problema com meu convênio de saúde e fiquei birutinha. OK, vamos tentar voltar pro corpo, né?  
  
Acho que teríamos que fazer:
* Comparar os sem identificação (SD) entre o drenagem e o correntes estimadas
* Aquele processo do Henrique tanto no drenagem (que ele já fez, mas quando for pro outro github eu vou dar uma mudada)... (? quanto no de correntes estimadas do Elias)
* Pegar os pontinhos no drenaghenrique e fazer um simmetrical difference com os pontos do correntes_estimadas...
* ... aí eu preciso conferir, né (ainda n sei como)

In [1]:
import geopandas as gpd
import pandas as pd
from os.path import join
from tqdm import tqdm

In [2]:
def testar_gdf(gdf):
    print(
    f'Shape: {gdf.shape};\n'+
    f'\nSample:{gdf.sample()}'
)

Ok, vamos começar do começo!
# A. Comparar drenagem ([GeoSampa](http://wfs.geosampa.prefeitura.sp.gov.br/geoserver/geoportal/wfs?version=1.0.0&request=GetFeature&outputFormat=SHAPE-ZIP&typeName=geoportal:drenagem)) e correntes estimadas ([Elias](https://github.com/sepep-pmsp/siiau/blob/master/projects/urbis/assets/silver/parquet_aguas_correntes_estimadas.py))

In [3]:
drenageo = gpd.read_file(
    join(
        'data',
        'drenagem.zip'
    )
)
print(
    f'Shape: {drenageo.shape};\n'+
    f'\nSample:{drenageo.sample()}'
)

## Como as correntes estimadas tem a id como int, vou transformar a id do drenageo em int tbm
drenageo['cd_identif'].astype('int', copy=False)

Shape: (27611, 15);

Sample:      cd_identif cd_tipo_ac tx_tipo_ac cd_numero_ nm_bairro nm_acident  \
7191      6803.0         ND       None          1      None         SD   

      qt_comprim  cd_tipo_cu                nm_tipo_cu nm_via_pro nm_descrit  \
7191  147.433694        11.0  Trecho em estado natural       None       None   

               nm_tipo_tr dt_atualiz cd_usuario  \
7191  Trecho a céu aberto 2025-01-03       None   

                                               geometry  
7191  LINESTRING (327594.293 7349184.621, 327584.132...  


0            1
1        26125
2            2
3        26126
4        26127
         ...  
27606    27596
27607    27597
27608    27598
27609    27599
27610    27600
Name: cd_identif, Length: 27611, dtype: int64

Provavelmente eles são iguais mesmo, só que as correntes estimadas são polígonos ao invés de linhas  
# B. Conferir SDs
Vamos conferir quais os 'nm_acidentes' que aparecem e se tem SD ou outra forma de dados vazios nesse recorte

Pelo que eu vi e conversei com o Mauryas: ```Considere os tipos para efeito de continuidade. Só não gere a nascente se esses tipos desconhecidos forem os finais de um curso.```  
Sabe o que isso quer dizer também? Que não dá pra eu separar por nomes igual o Henrique fez...

In [4]:
drenageo.sample(3) # o melhor era aqui ser um drenageo_in_estim, né, mas ok, vamos trabalhar só com o drenageo por enquanto

,cd_identif,cd_tipo_ac,tx_tipo_ac,cd_numero_,nm_bairro,nm_acident,qt_comprim,cd_tipo_cu,nm_tipo_cu,nm_via_pro,nm_descrit,nm_tipo_tr,dt_atualiz,cd_usuario,geometry
4908,26257.0,ND,None,2,JARDIM SAO GONCALO,SD,124.353893,11.0,Trecho em estado natural,MORAIS LOBO,None,Trecho a céu aberto,2025-01-03,None,"LINESTRING (351792.354 7389909.46, 351791.903 ..."
11490,10878.0,RIO,RIO,6,S/B,RIO CAPIVARI,296.334891,11.0,Trecho em estado natural,BELA VISTA,Rio Capivari,Trecho a céu aberto,2025-01-03,None,"LINESTRING (323966.784 7349827.336, 323959.388..."
12551,11873.0,RIB,RIBEIRAO,3,S/B,RIBEIRAO DOS PERUS,40.874839,11.0,Trecho em estado natural,S/N,Ribeirão dos Perus,Trecho a céu aberto,2025-01-03,None,"LINESTRING (321445.679 7408041.517, 321446.576..."


O que eu acho que dá pra fazer:  
* Dar um buffer, ou nos pontos, ou na linha
* Considerar, da lista com mesmo nome, qual é o que tem dois pontos touching um outro próximo com mesmo nome
* Deverão haver 2 sem touching, o mais alto no index é considerado o 1
* Faz aquele negócio que o Henrique ensinou, mas precisamos ter certeza de que vai terminar ou em A ou em B
* __A)__ total de linhas com nome -1
* __B)__ total de linhas com nome, mas garantir que nenhum ponto do final encoste em nenhum ponto do inicial (caso numer_de_segmentos>2)

Agora, sobre os SDs, como eu posso resolver?  
Talvez seja uma boa usar shortest line com este, mas não com os pontos...  
ANTES DE TUDO eu vou dar um explore com cores diferentes, já tendo a linha unificada dos nomes traçada. Aí eu consigo ter um panorama

Pelo que eu falei com o Mauryas, vamos ter que usar essas sd sim, né, mas vamos com calma
* Começamos traçando os rios com nome
* Plotamos os com nome por baixo e os sd de outra cor por cima
* Vemos qual o padrão dos SD e como prosseguir a partir daí

# B-1. Encontrar o Primeiro de cada nome e o último de cada nome
* a) vamos começar selecionando só um nome, pra testar
* b) dando certo a gente faz a função geral

In [5]:
gdf= drenageo[[
    'cd_identif', 
    'cd_tipo_ac', 
    'cd_tipo_cu', 
    'nm_acident', 
    'geometry'
]]

## B-1. a) Selecionar 1 para teste

In [6]:
nomes = gdf.loc[gdf['nm_acident']!='SD', 'nm_acident'].unique()
nomes= pd.Series(nomes)

nome_1 = nomes.sample(1, ignore_index=True)

one = gdf.loc[gdf['nm_acident']==nome_1[0]]

In [7]:
one.shape

(11, 5)

In [8]:
one_buff = one.copy()
one_buff['geometry']=one['geometry'].buffer(0.5)

In [9]:
one_buff.sample(1)

,cd_identif,cd_tipo_ac,cd_tipo_cu,nm_acident,geometry
1112,1026.0,RIO,10.0,RIO DAS PEDRAS,"POLYGON ((327118.277 7374139.487, 327139.374 7..."


In [10]:
# Verifica se a geometria no índice 'ix' intersecta com a geometria no índice 'iy'
#one_buff.loc[ix, 'geometry'].intersects(one_buff.loc[iy, 'geometry'])

outras_geoms= gpd.GeoDataFrame
for i, row in one_buff.iterrows():
    outras_geoms = one_buff.loc[one_buff.index!=i]
    intersecs_bool = row.geometry.intersects(outras_geoms.geometry)
    print(f'Nome:{row['nm_acident']}\n row: {i}\n {intersecs_bool}')
    if len(intersecs_bool.loc[intersecs_bool == True])<2:
        print('EXTREMIDADE!!!!!!')
    else: 
        print('keep on mooving')

Nome:RIO DAS PEDRAS
 row: 828
 1112     False
2745     False
5119     False
5752     False
11585     True
13266    False
13375    False
13739     True
15512    False
17989    False
Name: geometry, dtype: bool
keep on mooving
Nome:RIO DAS PEDRAS
 row: 1112
 828      False
2745      True
5119     False
5752     False
11585    False
13266    False
13375    False
13739    False
15512    False
17989     True
Name: geometry, dtype: bool
keep on mooving
Nome:RIO DAS PEDRAS
 row: 2745
 828      False
1112      True
5119      True
5752     False
11585    False
13266    False
13375    False
13739    False
15512    False
17989    False
Name: geometry, dtype: bool
keep on mooving
Nome:RIO DAS PEDRAS
 row: 5119
 828      False
1112     False
2745      True
5752     False
11585    False
13266     True
13375    False
13739    False
15512    False
17989    False
Name: geometry, dtype: bool
keep on mooving
Nome:RIO DAS PEDRAS
 row: 5752
 828      False
1112     False
2745     False
5119     False
11585

Agora a gente só precisa selecionar os que são extremidades e determinar que eles NÃO PODEM SE ENCONTRAR!!

OBS: TEM CASOS COM MAIS DE 2 EXTREMIDADES?
RIBEIRAO BROCADO

# B-1. b) Fazer a função geral
Agora o desafio vai ser ver como fazer isso sem ser um for dentro de outro...

In [11]:
#nomes já foi declarado ali em cima, no B-1. a)
#idem pra outras_geoms
gdf_buffer = gdf.copy()
gdf_buffer['geometry']=gdf['geometry'].buffer(10)
lista_extremidades=[]

for i, row in (
    gdf_buffer
    .loc[gdf_buffer['nm_acident']!='SD']
    .iterrows()
):
    outras_geoms = (
        gdf_buffer.loc[
            (gdf_buffer['nm_acident']==row['nm_acident']) 
            & (gdf_buffer.index!=i)
        ]
    )
    intersecs_bool = row.geometry.intersects(
        outras_geoms.geometry
    )
    if len(intersecs_bool.loc[intersecs_bool == True])<2:
        lista_extremidades.append(row)
extremidades = gpd.GeoDataFrame(lista_extremidades, crs=gdf.crs)


... Ok, acho que deu certo!  
Agora eu só preciso resolver o problema das extremidades falsas. Meu achismo?:
* tem SDs entre esses pedações de rios
* É bem onde tem um afluente próximo, indo se conectar

In [12]:
extremidades['nm_acident'].value_counts()

nm_acident
RIBEIRAO DOS COUROS         4
CORREGO FREZA               3
CORREGO AGUA DOS BRANCOS    3
CORREGO RAPADURA            2
CORREGO NOVO MUNDO          2
                           ..
CORREGO CAMPANELA           1
CORREGO JACUI               1
CORREGO SANTO ANTONIO       1
CORREGO PQ. MALAGONI        1
CORREGO DO ENGENHO          1
Name: count, Length: 327, dtype: int64

Pera, mas não devia ter nenhum que aparece só uma vez... devia? Isso não faz sentido...  
AAAAAH, são afluentes! ... eu não lembro se eles contavam como nascentes possíveis... eu vou precisar perguntar.

## visualizar únicos
nomes_1x=(
    extremidades['nm_acident']
    .value_counts()
    [extremidades['nm_acident']
    .value_counts() == 1]
)

m= gdf.explore(color='red')
gdf_buffer.loc[gdf['nm_acident'].isin(nomes_1x.index)].explore(
    m=m
)

In [13]:
nomes_mais2x = (
    extremidades['nm_acident']
    .value_counts()
    [extremidades['nm_acident']
    .value_counts() >2]
)

In [14]:
len(nomes_mais2x)

3

## Visualizar os que aparecem mais que 2 vezes

m= gdf.explore(color='red')
gdf_buffer.loc[gdf['nm_acident'].isin(nomes_mais2x.index)].explore(
    m=m
)

Vou começar só aumentando de meio metro pra um metro e ver se muda alguma coisa
* 0.5m: 118
* 1m: 114
* 2m: 109
* 5m: 30
* 10m: 3  
Vou tentar zerar esses...
# TO DO: mas precisará ainda comparar se não causou algum problema na quantidade de únicos e de apenas 2 